# Exercise 6 - AI Agents

_By Abigail Hayes_

In this exercise we will explore using LLMs as agents. We will focus on open-weight models that can be run locally (if you have a GPU available).

Note that the models we will use in this exercise are significantly larger (>1B parameters) than, e.g., distilbert (67M parameters). <br>
**This means that you will likely need to run this notebook on the BWUniCluster3.0 or on Google Colab to have enough GPU memory and compute.**

This exercise consists of the following parts:
1. Agent setup
2. Building a tool
3. Memory

## Setup

Make sure to select the same kernel as last week to have the previously added modules. We also need the following modules:
* smolagents - for handling agent implementation

In [ ]:
%pip install 'smolagents[toolkit, transformers]' 'flash-linear-attention'

# 1. Agent setup

Setup based on the [smolagents documentation](https://huggingface.co/docs/smolagents/index).

First we will set up a basic coding agent, using the same huggingface model as last week. It is also possible to [access other types of models](https://huggingface.co/docs/smolagents/examples/using_different_models).


In [ ]:
from smolagents import CodeAgent, TransformersModel

model = TransformersModel(model_id="Qwen/Qwen3.5-4B")

agent = CodeAgent(tools=[], model=model)

We have not yet provided the model with any tools, but it will instead write and execute code itself.

In [ ]:
result = agent.run("Calculate the sum of numbers from 1 to 10")
print(result)

Within the 'toolkit', we already have access to some tools that have been created for us. Let's try allowing the agent to search the internet.

In [ ]:
from smolagents import CodeAgent, TransformersModel, DuckDuckGoSearchTool

model = TransformersModel(model_id="Qwen/Qwen3.5-4B")

agent = CodeAgent(tools=[DuckDuckGoSearchTool()], 
                  model=model)

In [ ]:
result = agent.run("What is the current weather in Paris?")
print(result)

## 2. Building a tool

In fact, we can give the agent access to web search by enabling access to all 'base_tools'. For a code agent, this adds the DuckDuckGo web search and speech-to-text capabilities.

In [ ]:
from smolagents import CodeAgent, TransformersModel

model = TransformersModel(model_id="Qwen/Qwen3.5-4B")

agent = CodeAgent(tools=[], 
                  model=model,
                 add_base_tools=True)

In [ ]:
result = agent.run("Who is the president of Germany?")
print(result)

We can also manually call tools, to see what information they would provide to the agent.

In [ ]:
from smolagents import WebSearchTool

search_tool = WebSearchTool()
print(search_tool("Who's the current president of Germany?"))

In previous weeks, we have learnt basic text analysis methods. Let us now give the agent access to some tools that would help it to analyse text.

### Text statistics

_Text from: [www.bundespraesident.de](https://www.bundespraesident.de/SharedDocs/Reden/EN/Frank-Walter-Steinmeier/Reden/2026/261008-State-banquet-Paris.html)_

In [ ]:
from smolagents import tool

@tool
def text_stats(text: str) -> dict:
    """Analyse a text and return basic statistics.

    Args:
        text: The text to analyse.

    Returns:
        A dictionary containing character, word, and line counts.
    """
    return {
        "characters": len(text),
        "words": len(text.split()),
        "lines": len(text.splitlines()),
    }


In [ ]:
from smolagents import CodeAgent, TransformersModel

model = TransformersModel(model_id="Qwen/Qwen3.5-4B")

agent = CodeAgent(tools=[text_stats], 
                  model=model,
                 add_base_tools=True)

In [ ]:
result = agent.run("What are the basis statistics of the following text? 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal President said.'")
print(result)


But if we ask a more specific question, where the result can be extracted from our tool, the agent may need specific guidance to know to use it.

In [ ]:
result = agent.run("How many words are there in the following text? 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal President said.'")
print(result)

With a small hint, the agent can use the provided function.

In [ ]:
result = agent.run("How many words are there in the following text (use the text statistics)? 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal President said.'")
print(result)

Alternatively, we can try to improve the tool description, so that the agent realises how it can be used.

In [ ]:
from smolagents import tool

@tool
def text_stats(text: str) -> dict:
    """Use to return the number of characters, number of words and number of lines in a text.

    Args:
        text: The text to analyse.

    Returns:
        A dictionary containing character, word, and line counts.
    """
    return {
        "characters": len(text),
        "words": len(text.split()),
        "lines": len(text.splitlines()),
    }

In [ ]:
agent = CodeAgent(tools=[text_stats], 
                  model=model,
                 add_base_tools=True)

result = agent.run("How many words are there in the following text? 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal President said.'")
print(result)

## Task 1

We will do some further analysis of the text.

1. Ask the agent to count how many times the letter 'f' appears in the text.
2. Build a tool which counts how often each letter appears in the text.
3. Get the agent to use this tool to find out how many times the letter 'f' appears in the text.
4. How do the two responses differ? What might be the advantages or disadvantages of using tools for this task?


## 3. Memory

We also generally want agents to be able to work with a memory. Since smolagents aims to be lightweight, the memory system is lightweight and only persists through a single call. Other frameworks such as Autogen or Langchain have more elaborate memory systems as default. However, we can instead use a tool for the purpose.

In [ ]:
from MemoryBank import MemoryBankTool

memory_bank_tool = MemoryBankTool(
    memory_bank_dir_path=f"./memory_bank",
)

memory_bank_instruction = """ Use the Memory Bank tool to keep track of your work. You maintain a structured Memory Bank to ensure continuity and coherence across your problem-solving steps:
  1. Project Overview: Core objectives, scope, and user-defined goals for the current task.
  2. Active Progress Tracking: Current tasks, recent changes, and next steps.
  Remember to keep them updated. """

agent = CodeAgent(tools=[memory_bank_tool], 
                  model=model,
                 add_base_tools=True,
                 instructions=[memory_bank_instruction])

In [ ]:
request = """
Do a deep research about vaccines, remember to use your Memory Bank:
- find the 5 most common vaccines used throughout the world nowadays
- find source material about the safety profile of each one of these vaccines
- find recommendations of application of these vaccines for public authorities, including schedules, target groups and other
similar relevant information
- add some historical context about each vaccine, including the year of introduction and the disease it is meant to prevent
- produce a final report with the relevant content summarized in a single markdown document, with references cited where needed
"""

response = agent.run(request)
print(response)

In [ ]:
with open(f"./memory_bank/vaccine_report.md", "w") as f:
    f.write(response)

We can continue to reuse the information that the agent has stored.

In [ ]:
request = """
Using only the information stored in your memory, when was the TB vaccine first developed?"""

response = agent.run(request)
print(response)

## Task 2

Choose a topic from the options below (or come up with your own) for your agent to research. Then ask it to answer the accompanying questions from it's memory.

### Wrestling at the 1924 Olympics
1. Which country won the most wrestling medals at the 1924 Olympics?
2. How many wrestling events were there for women at the 1924 Olympics?
3. Where did you find the information about wrestling at the 1924 Olympics?

### Kazakhstan
1. Why was the capital city of Kazakhstan changed?
2. Where did you find out information about Kazakhstan?
3. What is the most important information for me to know about Kazakhstan?